# CET6 解析PDF批量转Markdown (MinerU)

将答案/解析PDF通过MinerU转为高质量Markdown，用于后续提取听力答案。

In [1]:
# Cell 1: 安装 MinerU
!pip install --upgrade pip -q
!pip install "mineru[all]>=3.1.3" -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have openteleme

In [7]:
# 查看 Colab 当前 CUDA 版本
!nvcc --version
!ls /usr/local/cuda/lib64/libcudart*

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
/usr/local/cuda/lib64/libcudart.so
/usr/local/cuda/lib64/libcudart.so.12
/usr/local/cuda/lib64/libcudart.so.12.8.90
/usr/local/cuda/lib64/libcudart_static.a


In [2]:
# Cell 2: 验证安装
!mineru -v

mineru, version 3.2.1


In [3]:
# Cell 3: 挂载 Google Drive 并定位PDF
from google.colab import drive
drive.mount('/content/drive')

import os, re, shutil

DRIVE_BASE = '/content/drive/MyDrive/CET6-Resources'

# 需要处理的文件（目录/文件名）
pdf_list = [
    ('CET6_2016.12', '2016年12月六级（第1套）答案及解析.pdf'),
    ('CET6_2016.12', '2016年12月六级（第2套）答案及解析.pdf'),
    ('CET6_2017.06', '2017.06英语六级考试第1套解析.pdf'),
    ('CET6_2017.06', '2017.06英语六级考试第2套解析.pdf'),
    ('CET6_2017.12', '2017.12英语六级考试第1套解析.pdf'),
    ('CET6_2017.12', '2017.12英语六级考试第2套解析.pdf'),
    ('CET6_2018.12', '2018年12月6级答案解析第一套.pdf'),
    ('CET6_2019.06', '2019年06月真题解析第1套.pdf'),
    ('CET6_2019.06', '2019年06月真题解析第2套.pdf'),
    ('CET6_2020.09', '2020.09英语六级考试第1套解析.pdf'),
    ('CET6_2021.06', '2021.06英语六级答案解析第2套.pdf'),
    ('CET6_2022.06', '2022.06英语六级真题解析第2套 .pdf'),
    ('CET6_2022.12', '2022.12六级真题第1套答案及详解.pdf'),
    ('CET6_2023.06', '2023.06六级真题第1套详解.pdf'),
    ('CET6_2023.12', '2023.12英语六级解析第1套(1).pdf'),
    ('CET6_2023.12', '2023.12英语六级解析第2套(1).pdf'),
    ('CET6_2024.12', '2024.12英语六级解析第1套.pdf'),
    ('CET6_2024.12', '2024.12英语六级解析第2套.pdf'),
    ('CET6_2025.06', '2025.06英语六级解析第1套.pdf'),
    ('CET6_2025.06', '2025.06英语六级解析第2套.pdf'),
]

def detect_set_number(filename):
    """从文件名提取套数"""
    chinese_nums = {'一': 1, '二': 2, '三': 3}
    m = re.search(r'第\s*(\d)\s*套', filename)
    if m:
        return int(m.group(1))
    m = re.search(r'第([一二三])套', filename)
    if m:
        return chinese_nums[m.group(1)]
    m = re.search(r'[（(]卷([一二三1-3])[)）]', filename)
    if m:
        ch = m.group(1)
        return chinese_nums.get(ch, int(ch) if ch.isdigit() else 1)
    return 1

def get_exam_date(dir_name):
    """从目录名提取 YYYY.MM"""
    m = re.match(r'CET6_(\d{4}\.\d{2})', dir_name)
    return m.group(1) if m else None

# 复制PDF到本地，同时建立 PDF→目标文件名 的映射
os.makedirs('pdfs', exist_ok=True)
file_mapping = {}  # local_pdf_path -> target_md_name
found, missing = [], []

for dir_name, fname in pdf_list:
    src = os.path.join(DRIVE_BASE, dir_name, fname)
    if os.path.exists(src):
        local_path = f'pdfs/{fname}'
        shutil.copy2(src, local_path)
        # 计算目标文件名
        date = get_exam_date(dir_name)
        set_num = detect_set_number(fname)
        # 判断该目录下是否有多套
        dir_pdfs = [d for d, f in pdf_list if d == dir_name]
        if len(dir_pdfs) > 1:
            target_name = f'CET6_{date}_set{set_num}_transcript.md'
        else:
            target_name = f'CET6_{date}_transcript.md'
        file_mapping[fname] = (dir_name, target_name)
        found.append(fname)
    else:
        missing.append(f'{dir_name}/{fname}')

print(f'已复制 {len(found)} 个PDF到本地')
if missing:
    print(f'\n未找到 {len(missing)} 个文件:')
    for f in missing:
        print(f'  ✗ {f}')

print('\n文件名映射:')
for pdf_name, (dir_name, target) in sorted(file_mapping.items(), key=lambda x: x[1]):
    print(f'  {pdf_name}  →  {dir_name}/{target}')

Mounted at /content/drive
已复制 20 个PDF到本地

文件名映射:
  2016年12月六级（第1套）答案及解析.pdf  →  CET6_2016.12/CET6_2016.12_set1_transcript.md
  2016年12月六级（第2套）答案及解析.pdf  →  CET6_2016.12/CET6_2016.12_set2_transcript.md
  2017.06英语六级考试第1套解析.pdf  →  CET6_2017.06/CET6_2017.06_set1_transcript.md
  2017.06英语六级考试第2套解析.pdf  →  CET6_2017.06/CET6_2017.06_set2_transcript.md
  2017.12英语六级考试第1套解析.pdf  →  CET6_2017.12/CET6_2017.12_set1_transcript.md
  2017.12英语六级考试第2套解析.pdf  →  CET6_2017.12/CET6_2017.12_set2_transcript.md
  2018年12月6级答案解析第一套.pdf  →  CET6_2018.12/CET6_2018.12_transcript.md
  2019年06月真题解析第1套.pdf  →  CET6_2019.06/CET6_2019.06_set1_transcript.md
  2019年06月真题解析第2套.pdf  →  CET6_2019.06/CET6_2019.06_set2_transcript.md
  2020.09英语六级考试第1套解析.pdf  →  CET6_2020.09/CET6_2020.09_transcript.md
  2021.06英语六级答案解析第2套.pdf  →  CET6_2021.06/CET6_2021.06_transcript.md
  2022.06英语六级真题解析第2套 .pdf  →  CET6_2022.06/CET6_2022.06_transcript.md
  2022.12六级真题第1套答案及详解.pdf  →  CET6_2022.12/CET6_2022.12_transcript.md
  2023.06六级真题第1

In [12]:
# Cell 4: 批量转换
# 使用 pipeline 后端（兼容性好，CPU/GPU均可）
!pip install vllm==0.8.4 --no-deps
import os, glob

pdf_files = sorted(glob.glob('pdfs/*.pdf'))
batch_size = 3

for i in range(0, len(pdf_files), batch_size):
    batch = pdf_files[i:i+batch_size]
    batch_names = [os.path.basename(f) for f in batch]
    print(f'\n=== Batch {i//batch_size + 1}: {batch_names} ===')
    for pdf in batch:
        !mineru -p "{pdf}" -o output/ -b pipeline


=== Batch 1: ['2016年12月六级（第1套）答案及解析.pdf', '2016年12月六级（第2套）答案及解析.pdf', '2017.06英语六级考试第1套解析.pdf'] ===
2026-05-30 09:02:38.881 | INFO     | mineru.cli.client:run_orchestrated_cli:901 - Started local mineru-api at http://127.0.0.1:54811
2026-05-30 09:02:42.592 | INFO     | __main__:create_app:234 - Request concurrency limited to 3
Start MinerU FastAPI Service: http://127.0.0.1:54811
API documentation: http://127.0.0.1:54811/docs
INFO:     Started server process [9688]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:54811 (Press CTRL+C to quit)
2026-05-30 09:02:42.904 | INFO     | mineru.cli.client:run_planned_task:781 - Submitting batch 1/1 | 1 document, 64 pages in this batch | 64 pages total | task#1 [2016年12月六级（第1套）答案及解析]
2026-05-30 09:02:53.724843: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.


In [13]:
# Cell 5: 查看输出结构
import os

print('输出目录结构：')
for root, dirs, fnames in os.walk('output'):
    # 只显示md文件
    for f in fnames:
        if f.endswith('.md'):
            path = os.path.join(root, f)
            size = os.path.getsize(path)
            print(f'  {path} ({size/1024:.1f} KB)')

输出目录结构：
  output/2019年06月真题解析第2套/auto/2019年06月真题解析第2套.md (230.7 KB)
  output/2017.12英语六级考试第2套解析/auto/2017.12英语六级考试第2套解析.md (234.5 KB)
  output/2018年12月6级答案解析第一套/auto/2018年12月6级答案解析第一套.md (64.2 KB)
  output/2022.06英语六级真题解析第2套 /auto/2022.06英语六级真题解析第2套 .md (41.5 KB)
  output/2020.09英语六级考试第1套解析/auto/2020.09英语六级考试第1套解析.md (247.9 KB)
  output/2025.06英语六级解析第2套/auto/2025.06英语六级解析第2套.md (236.5 KB)
  output/2017.12英语六级考试第1套解析/auto/2017.12英语六级考试第1套解析.md (239.3 KB)
  output/2023.06六级真题第1套详解/auto/2023.06六级真题第1套详解.md (64.5 KB)
  output/2021.06英语六级答案解析第2套/auto/2021.06英语六级答案解析第2套.md (64.5 KB)
  output/2023.12英语六级解析第1套(1)/auto/2023.12英语六级解析第1套(1).md (75.2 KB)
  output/2016年12月六级（第1套）答案及解析/auto/2016年12月六级（第1套）答案及解析.md (234.1 KB)
  output/2024.12英语六级解析第1套/auto/2024.12英语六级解析第1套.md (66.0 KB)
  output/2024.12英语六级解析第2套/auto/2024.12英语六级解析第2套.md (65.7 KB)
  output/2016年12月六级（第2套）答案及解析/auto/2016年12月六级（第2套）答案及解析.md (234.2 KB)
  output/2019年06月真题解析第1套/auto/2019年06月真题解析第1套.md (233.6 KB)
  output/2017.06英语六级考试第1套解析

In [15]:
extra_pdfs = [
      ('CET6_2019.12', '2019.12英语六级考试解析第1套.pdf'),
      ('CET6_2019.12', '2019.12英语六级考试解析第2套.pdf'),
      ('CET6_2020.12', '2020.12英语六级考试第1套解析.pdf'),
      ('CET6_2020.12', '2020.12英语六级考试第2套解析.pdf'),
  ]
for dir_name, fname in extra_pdfs:
      src = os.path.join(DRIVE_BASE, dir_name, fname)
      if os.path.exists(src):
          shutil.copy2(src, f'pdfs/{fname}')
          !mineru -p "pdfs/{fname}" -o output/ -b pipeline

          date = get_exam_date(dir_name)
          set_num = detect_set_number(fname)
          target_name = f'CET6_{date}_set{set_num}_transcript.md'
          file_mapping[fname] = (dir_name, target_name)
          print(f'  ✓ {fname} → {target_name}')

2026-05-30 09:58:42.284 | INFO     | mineru.cli.client:run_orchestrated_cli:901 - Started local mineru-api at http://127.0.0.1:41621
2026-05-30 09:58:45.857 | INFO     | __main__:create_app:234 - Request concurrency limited to 3
Start MinerU FastAPI Service: http://127.0.0.1:41621
API documentation: http://127.0.0.1:41621/docs
INFO:     Started server process [25344]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:41621 (Press CTRL+C to quit)
2026-05-30 09:58:46.304 | INFO     | mineru.cli.client:run_planned_task:781 - Submitting batch 1/1 | 1 document, 16 pages in this batch | 16 pages total | task#1 [2019.12英语六级考试解析第1套]
2026-05-30 09:58:50.590097: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with t

In [16]:
# Cell 6: 重命名并保存回Google Drive对应目录
import shutil

# MinerU输出的md文件名是 {pdf_stem}.md（去掉.pdf后缀）
# 需要映射回目标文件名并放到对应的Drive子目录

saved = []
for pdf_name, (dir_name, target_name) in file_mapping.items():
    # MinerU 输出路径: output/{pdf_stem}/{backend}/{pdf_stem}.md
    pdf_stem = os.path.splitext(pdf_name)[0]
    # 搜索实际输出的md文件
    md_found = None
    for root, dirs, fnames in os.walk(f'output/{pdf_stem}'):
        for f in fnames:
            if f.endswith('.md') and not f.endswith('_layout.md'):
                md_found = os.path.join(root, f)
                break
        if md_found:
            break

    if md_found:
        # 保存到Drive对应目录
        dst_dir = os.path.join(DRIVE_BASE, dir_name)
        dst_path = os.path.join(dst_dir, target_name)
        shutil.copy2(md_found, dst_path)
        saved.append(f'{dir_name}/{target_name}')
        print(f'  ✓ {dir_name}/{target_name}')
    else:
        print(f'  ✗ 未找到输出: {pdf_name}')

print(f'\n共保存 {len(saved)} 个文件到 Google Drive')

  ✓ CET6_2016.12/CET6_2016.12_set1_transcript.md
  ✓ CET6_2016.12/CET6_2016.12_set2_transcript.md
  ✓ CET6_2017.06/CET6_2017.06_set1_transcript.md
  ✓ CET6_2017.06/CET6_2017.06_set2_transcript.md
  ✓ CET6_2017.12/CET6_2017.12_set1_transcript.md
  ✓ CET6_2017.12/CET6_2017.12_set2_transcript.md
  ✓ CET6_2018.12/CET6_2018.12_transcript.md
  ✓ CET6_2019.06/CET6_2019.06_set1_transcript.md
  ✓ CET6_2019.06/CET6_2019.06_set2_transcript.md
  ✓ CET6_2020.09/CET6_2020.09_transcript.md
  ✓ CET6_2021.06/CET6_2021.06_transcript.md
  ✓ CET6_2022.06/CET6_2022.06_transcript.md
  ✓ CET6_2022.12/CET6_2022.12_transcript.md
  ✓ CET6_2023.06/CET6_2023.06_transcript.md
  ✓ CET6_2023.12/CET6_2023.12_set1_transcript.md
  ✓ CET6_2023.12/CET6_2023.12_set2_transcript.md
  ✓ CET6_2024.12/CET6_2024.12_set1_transcript.md
  ✓ CET6_2024.12/CET6_2024.12_set2_transcript.md
  ✓ CET6_2025.06/CET6_2025.06_set1_transcript.md
  ✓ CET6_2025.06/CET6_2025.06_set2_transcript.md
  ✓ CET6_2019.12/CET6_2019.12_set1_transcript.md
 